# IaC on AWS

## AWS Account

- Concepts introduced in the lesson are valid to any AWS account (including AWS free tier)
- We will use a module in the AWS Academy Lab: [ACAv3EN-US-LTI13-163301 - Module 11: Automating Your Architecture](https://awsacademy.instructure.com/courses/163301/modules/items/15994616) 
- Specifically the LAB <https://awsacademy.instructure.com/courses/163301/assignments/1951546?module_item_id=15994666>
- Start the Lab

### AWS CLI

- Download and install AWS CLI <https://docs.aws.amazon.com/cli/latest/userguide/getting-started-install.html>
- Copy credentials from the *AWS Details* panel: ```aws configure # or paste directly into ~/.aws/credentials```

### SSH Key

AWS Academy provides a pre-created key pair and a `.pem` file to download.

- Download `labsuser.pem` from the lab page
- Secure the key locally: ```chmod 600 labsuser.pem ```

:::{.callout-note .fragment}
The key pair name in AWS is `vockey` — the file is `labsuser.pem`
:::

## Opentofu

::::{.columns}

::: {.fragment .column width="30%"}

**providers.tf**
```hcl
terraform {
  required_providers {
    aws = {
      source  = "hashicorp/aws"
      version = "~> 5.0"
    }
  }
}

provider "aws" {
  region = var.aws_region
}
```

**variables.tf**
```hcl
variable "aws_region" {
  default = "us-east-1"
}
variable "instance_type" {
  default = "t2.micro"
}
variable "key_pair_name" {
  default = "vockey"
}
```

:::

::: {.fragment .column width="70%"}

**main.tf**
```hcl
data "aws_ami" "ubuntu" {
  most_recent = true
  owners      = ["099720109477"]
  filter {
    name   = "name"
    values = ["ubuntu/images/hvm-ssd-gp3/ubuntu-noble-24.04-amd64-*"]
  }
}
resource "aws_security_group" "ssh" {
  name = "lab-ssh-sg"
  ingress {
    from_port   = 22
    to_port     = 22
    protocol    = "tcp"
    cidr_blocks = ["0.0.0.0/0"]
  }
}

resource "aws_instance" "manager" {
  ami                    = data.aws_ami.ubuntu.id
  instance_type          = var.instance_type
  key_name               = var.key_pair_name
  vpc_security_group_ids = [aws_security_group.ssh.id]
  tags = { Name = "lab-cloud-systems-2026" }
}
```

**outputs.tf**
```hcl
output "instance_public_ip" {
  value = aws_instance.manager.public_ip
}
```

:::

::::


### LAB - Create an EC2 instance using Opentofu

:::{.fragment}
**Goto** to the lab directory ```cd lab/aws-opentofu```
:::

:::{.fragment}
**Run** the code 
```bash
tofu init
tofu plan
tofu apply
```
:::

:::{.fragment}
**SSH** into the instance using the public IP from the output
:::

:::{.fragment}
**Destroy** when done:
```bash
tofu destroy
```
:::


## CloudFormation

::::{.columns}

::: {.fragment .column width="50%"}

**stack.yaml — Parameters & Resources**
```yaml
Parameters:
  InstanceType:
    Type: String
    Default: t2.micro
  KeyPairName:
    Type: String
    Default: vockey
  UbuntuAMI:
    Type: AWS::SSM::Parameter::Value<AWS::EC2::Image::Id>
    Default: /aws/service/canonical/ubuntu/server/24.04/...

Resources:
  SSHSecurityGroup:
    Type: AWS::EC2::SecurityGroup
    Properties:
      GroupDescription: Allow SSH inbound
      SecurityGroupIngress:
        - IpProtocol: tcp
          FromPort: 22
          ToPort: 22
          CidrIp: 0.0.0.0/0

  Manager:
    Type: AWS::EC2::Instance
    Properties:
      ImageId: !Ref UbuntuAMI
      InstanceType: !Ref InstanceType
      KeyName: !Ref KeyPairName
      SecurityGroupIds:
        - !GetAtt SSHSecurityGroup.GroupId
      Tags:
        - Key: Name
          Value: lab-cloud-systems-2026
```

:::

::: {.fragment .column width="50%"}

**stack.yaml — Outputs**
```yaml
Outputs:
  InstancePublicIP:
    Value: !GetAtt Manager.PublicIp

  SSHCommand:
    Value: !Sub "ssh -i ~/Downloads/labsuser.pem ubuntu@${Manager.PublicIp}"

  AWSCLIDescribe:
    Value: "aws ec2 describe-instances ..."
```
:::

::::

### LAB - Create an EC2 instance using CloudFormation

:::{.fragment}
**Goto** the lab directory ```cd lab/aws-cloudformation```
:::

:::{.fragment}
**Deploy** the stack
```bash
./deploy.sh
```
:::

:::{.fragment}
**SSH** into the instance using the public IP from the outputs table
:::

:::{.fragment}
**Destroy** when done:
```bash
./destroy.sh
```
:::


## OpenTofu vs CloudFormation

| Concept | OpenTofu (HCL) | CloudFormation (YAML) |
|---|---|---|
| **Language** | HCL (HashiCorp Config Language) | YAML / JSON |
| **Provider / vendor** | Multi-cloud (`hashicorp/aws`, `azure`, …) | AWS-only |
| **Input variables** | `variables.tf` | `Parameters` |
| **AMI lookup** | `data "aws_ami"` (dynamic query) | Passed as parameter (looked up via CLI) |
| **Security group** | `resource "aws_security_group"` | `AWS::EC2::SecurityGroup` |
| **EC2 instance** | `resource "aws_instance"` | `AWS::EC2::Instance` |
| **Cross-resource ref** | `aws_security_group.ssh.id` | `!GetAtt SSHSecurityGroup.GroupId` |
| **String interpolation** | `"${var.name}"` | `!Sub "${ResourceName.Attr}"` |
| **Outputs** | `outputs.tf` | `Outputs` block |
| **State tracking** | `terraform.tfstate` (local or remote) | Managed by CloudFormation service |
| **Deploy** | `tofu init && tofu apply` | `aws cloudformation deploy` |
| **Destroy** | `tofu destroy` | `aws cloudformation delete-stack` |
| **Dry run / preview** | `tofu plan` | `--no-execute-changeset` (change sets) |
| **Portability** | ✅ Works on any cloud | ❌ AWS only |
| **AWS integration** | Via API (needs credentials) | Native — no extra tooling |
